In [1]:
"""
本地测试脚本 - 不依赖 FastAPI

"""
import os
import sys
from dotenv import load_dotenv
from openai import OpenAI
from vector import (
    build_vector_db,
    hybrid_search,
    dense_search,
    sparse_search,
    get_doc_count,
    clear_all,
    router,
    rewrite,
    intent,
    rewrite_for_retrieval,
    resolve_placeholders
)

load_dotenv()

# ============ 配置 ============
REPORT_PATH =  "D:/github/rag_project/rag_财报/财报" 
API_KEY = os.getenv('DEEPSEEK_API_KEY')
API_BASE = os.getenv('DEEPSEEK_API_BASE')
MODEL = os.getenv('DEEPSEEK_MODEL')

client = OpenAI(api_key=API_KEY, base_url=API_BASE)

model = None
history = []  # 对话历史

def init_system():
    """初始化系统"""
    global model, history
    
    # 初始化向量数据库
    model = build_vector_db(REPORT_PATH, rebuild=False)
    
    # 系统提示词
    system_prompt = """你是一个专业的财务分析助手。\n
1. 请**仅根据**【本轮财报片段】来回答用户的问题。\n
2. 如果提供的片段中没有相关信息，请直接回答"根据提供的财报内容，我无法回答该问题"，绝不要编造数据。\n
3. 在回答的时候, 不需要输出分析的过程, 直接摆出关键数据然后回答用户问题\n
4. 不要开启思考模式。尽可能少消费token完成任务。"""
    
    # 清空历史并添加系统提示词
    history = [{"role": "system", "content": system_prompt}]
    
    # 显示状态
    doc_count = get_doc_count()
    print(f"[2/2] 初始化完成！")
    print(f"   - 当前知识库文本块数: {doc_count}")
    print(f"   - 财报文件: {REPORT_PATH}")
    print("=" * 60)


def test_search(query: str, mode: str = "hybrid", top_k: int = 5, show_sources: bool = True, source_filter: str = None):
    """
    测试检索功能

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 返回结果数
        show_sources: 是否显示检索到的原始片段
        source_filter: 指定检索的源文件
    """
    print(f"\n{'='*60}")
    print(f" 查询问题: {query}")
    print(f" 检索模式: {mode}")
    print(f" 返回数量: {top_k}")
    print("=" * 60)

    # 根据mode执行不同的检索
    if mode == "dense":
        results = dense_search(query, top_k=top_k, source_filter=source_filter)
        print(" 向量检索 (BGE向量)")
    elif mode == "sparse":
        results = sparse_search(query, top_k=top_k, source_filter=source_filter)
        print(" 关键词检索 (BM25)")
    else:  # hybrid
        results = hybrid_search(query, top_k=top_k, source_filter=source_filter)
        # 同时获取密集检索结果用于显示得分
        dense_results = dense_search(query, top_k=10, source_filter=source_filter)
        print(" 混合检索 (BGE向量 + BM25 + RRF)")

    # 显示检索结果
    print(f"\n 检索到 {len(results)} 条相关片段:")
    print("-" * 60)

    for i, (doc_id, chunk_text, score, source_file) in enumerate(results, 1):
            if mode == "dense":
                # 将距离转换为相似度：similarity = 1 / (1 + distance)
                similarity = 1 / (1 + score)
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
            elif mode == "sparse":
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  BM25得分: {score:.4f}")
            else:  # hybrid
                d_score = {d_id: s for d_id, _, s, _ in dense_results}.get(doc_id, float('inf'))
                similarity = 1 / (1 + d_score) if d_score != float('inf') else 0
                print(f"\n[片段 {i}] ID: {doc_id} | 来源: {source_file}")
                print(f"  RRF得分: {score:.4f} | 向量相似度: {similarity:.4f}")
            
            if show_sources:
                print(f"  内容: {chunk_text}")
    return results


def test_generation(query: str, mode: str = "hybrid", top_k: int = 5):
    """
    测试完整流程（意图识别 + 检索 + 生成）

    Args:
        query: 查询问题
        mode: 检索模式 (dense/sparse/hybrid)
        top_k: 检索返回数量
    """
    global history, last_retrieved_chunks
    
    # --- 1. 意图识别 ---
    user_intent = intent(query, history)
    
    if user_intent == "chat":
        # 闲聊：直接调用LLM回答
        print("\n检测到闲聊，直接回答...")
        
        # 保留系统提示词，添加用户问题
        chat_history = [history[0]] if history and history[0].get('role') == 'system' else []
        chat_history.append({"role": "user", "content": query})
        
        response = client.chat.completions.create(
            model=MODEL,
            extra_body={"thinking": {"type": "disabled"}},
            messages=chat_history,
            temperature=0.7
        )
        answer = response.choices[0].message.content
        
        # 更新历史（不包含财报片段）
        history.append({"role": "user", "content": query})
        history.append({"role": "assistant", "content": answer})
        
        print(f'\nAI 回答:\n{answer}')
        return
    
    # --- 2. Query Rewriting（处理代词、省略）---
    query = rewrite(query, history)
    
    # --- 3. LLM 智能路由 ---
    target_docs = router(query)
    
    # --- 4. 第二步改写 ---
    query = rewrite_for_retrieval(query)
    print(f'改写后用于检索的query:{query}')

    # --- 5. 检索逻辑：支持跨文档分别检索 ---
    all_results = []
    
    if target_docs == [None]:
        # 检索全部文档
        results = test_search(query, mode=mode, top_k=top_k, source_filter=None)
        all_results = results
    elif len(target_docs) == 1:
        # 单个文档
        results = test_search(query, mode=mode, top_k=top_k, source_filter=target_docs[0])
        all_results = results
    else:
        # 多个文档：分别检索每个文档，然后合并
        print(f" 跨文档查询，分别检索 {len(target_docs)} 个文档...")
        for doc in target_docs:
            print(f"\n  正在检索: {doc}")
            results = test_search(query, mode=mode, top_k=top_k, source_filter=doc)
            all_results.extend(results)
        
        # 按分数重新排序并取top_k
        if mode == "hybrid":
            all_results.sort(key=lambda x: x[2], reverse=True)
        elif mode == "dense":
            all_results.sort(key=lambda x: x[2])  # 距离越小越好
        elif mode == "sparse":
            all_results.sort(key=lambda x: x[2], reverse=True)
        
        # all_results = all_results[:top_k]
        print(f"\n 合并后取 top-{len(all_results)} 结果")

    if not all_results:
        print("\n 无法生成回答：未找到相关内容")
        return

    # --- 6. 解析占位符 ---
    print("\n 正在解析表格占位符...")
    all_results = resolve_placeholders(all_results)
    
    retrieved_contexts = []
    for _, text, _, source in all_results:
        # 去掉 .md 后缀
        clean_source = source.replace('.md', '')
        # 拼接成带有来源标识的区块
        context_block = f"【来源文件：{clean_source}】\n{text}"
        retrieved_contexts.append(context_block)
        
    context_text = "\n\n---\n\n".join(retrieved_contexts)

    # 删除历史中旧的财报片段
    history[:] = [msg for msg in history if not (msg.get('role') == 'system' and '【本轮财报片段】' in msg.get('content', ''))]
    
    # 添加新的财报片段和用户问题
    history.append({"role": "system", "content": f"【本轮财报片段】\n{context_text}"})
    history.append({"role": "user", "content": query})
    
    # 直接用 history 调用 LLM
    response = client.chat.completions.create(
        model=MODEL,    
        extra_body={"thinking": {"type": "disabled"}},
        messages=history,
        temperature=0.1
    )
    answer = response.choices[0].message.content
    
    history.append({"role": "assistant", "content": answer})
    
    print('\nAI 回答')
    print(f'AI回答: {answer}')

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
init_system()
query = '盛和资源和合诚股份，哪家公司2019年的研发投入总额占营业收入比例更高？'
results = test_generation(query, top_k=10)

Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\Arvin\AppData\Local\Temp\jieba.cache


数据库中已有 3273 条记录


Loading model cost 0.399 seconds.
Prefix dict has been built successfully.


[2/2] 初始化完成！
   - 当前知识库文本块数: 3273
   - 财报文件: D:/github/rag_project/rag_财报/财报
需要检索
🤖 LLM返回: ['盛和资源控股股份有限公司2019年年度报告.md', '合诚工程咨询集团股份有限公司2019年年度报告.md']
✅ 匹配到文档: ['盛和资源控股股份有限公司2019 年年度报告.md', '合诚工程咨询集团股份有限公司2019 年年度报告.md']
 检索优化rewrite: 盛和资源和合诚股份，哪家公司2019年的研发投入总额占营业收入比例更高？ → 盛和资源和合诚股份2019年研发投入总额占营业收入比例比较
改写后用于检索的query:盛和资源和合诚股份2019年研发投入总额占营业收入比例比较
 跨文档查询，分别检索 2 个文档...

  正在检索: 盛和资源控股股份有限公司2019 年年度报告.md

 查询问题: 盛和资源和合诚股份2019年研发投入总额占营业收入比例比较
 检索模式: hybrid
 返回数量: 10
 混合检索 (BGE向量 + BM25 + RRF)

 检索到 10 条相关片段:
------------------------------------------------------------

[片段 1] ID: 3024 | 来源: 盛和资源控股股份有限公司2019 年年度报告.md
  RRF得分: 0.0306 | 向量相似度: 0.7393
  内容: <html><body><table><tr><td>公司名称</td><td>公司类型</td><td>主要产品或服务</td><td>注册资本</td><td>总资产</td><td>营业收入</td><td>净利润</td></tr><tr><td rowspan="2">乐山盛和稀土股份 有限公司</td><td rowspan="2">控股子 公司</td><td rowspan="2">稀土氧化物、稀土 盐类、稀有稀土金 属</td><td rowspan="2"></td><td rowspan="2"></td><td rowspan="2">54,406.92</td><td rowspan="2">4,195.30</td></tr><tr><td>8,000

In [16]:
history

[{'role': 'user', 'content': '汇金股份的董事长是谁？'},
 {'role': 'assistant',
  'content': '根据你提供的财报片段，河北汇金机电股份有限公司（汇金股份）的法定代表人是**邢海平**。'},
 {'role': 'user', 'content': '邢海平持有汇金股份多少股份？'},
 {'role': 'assistant',
  'content': '根据你提供的财报片段，**没有直接披露邢海平持有汇金股份的具体股份数量**。\n\n不过，从片段中可以推断出以下信息：\n\n1.  **邢海平的身份**：他是邯郸市建设投资集团有限公司（简称“邯郸建投”）的董事长。而邯郸建投在2019年通过受让股份，成为了汇金股份的控股股东，邯郸市国资委成为实际控制人。\n2.  **股份持有关系**：邢海平作为邯郸建投的董事长，其个人是否直接持有汇金股份的股份，以及持有多少，在提供的这些片段中并未提及。通常，这类信息会在年报的“董事、监事、高级管理人员持股情况”部分详细列出，但该部分内容未包含在你提供的文本中。\n\n**结论**：仅凭现有片段，无法确定邢海平个人持有汇金股份的股份数量。如需准确数据，建议查阅汇金股份2019年年度报告全文中的“董事、监事和高级管理人员持股变动”章节。'},
 {'role': 'user', 'content': '邢海平在汇金股份任职多久了？'},
 {'role': 'assistant',
  'content': '根据你提供的财报片段，可以推断出邢海平在汇金股份的任职时间如下：\n\n1.  **首次任职**：他于**2018年12月**开始担任公司董事。\n2.  **职务变更**：\n    -   **2019年1月至2019年4月**：担任公司副董事长。\n    -   **2019年4月至今**（指报告期，即2019年年度报告发布时）：担任公司董事长。\n\n**结论**：截至2019年年度报告发布时（2020年4月），邢海平在汇金股份的任职时间约为**1年4个月**（从2018年12月至2020年4月）。'},
 {'role': 'system',
  'content': '【本轮财报片段】\n<html><body><table><tr><td

In [3]:
# 测试表格检索：查看用于检索的summary和用于回答的HTML
import psycopg
from pgvector.psycopg import register_vector
from vector import get_db_connection

# 执行一个会检索表格的查询
test_query = "合诚工程咨询集团的员工规模"

print("=" * 60)
print(f"测试查询: {test_query}")
print("=" * 60)

# 使用混合检索获取结果
from vector import hybrid_search

results = hybrid_search(test_query, top_k=10)

print("\n" + "=" * 60)
print("检索结果分析")
print("=" * 60)

# 连接数据库获取完整信息
conn = get_db_connection()
register_vector(conn)

for doc_id, chunk_text, score in results:
    print(f"\n{'='*60}")
    print(f"[片段 ID: {doc_id}] RRF得分: {score:.4f}")
    print("=" * 60)
    
    # 从数据库查询该记录的summary和table_id
    cur = conn.execute(
        "SELECT summary, table_id, source_file FROM financial_vectors WHERE id = %s",
        (doc_id,)
    )
    row = cur.fetchone()
    
    if row:
        db_summary, table_id, source_file = row
        
        print(f"\n来源文件: {source_file}")
        
        if table_id:
            print(f"表格ID: {table_id}")
            print(f"\n【用于检索的Summary】:")
            print("-" * 40)
            print(db_summary if db_summary else "(无summary)")
            
            print(f"\n【用于回答的HTML代码】:")
            print("-" * 40)
            print(chunk_text[:500] if len(chunk_text) > 500 else chunk_text)
            if len(chunk_text) > 500:
                print(f"\n... (共{len(chunk_text)}字符)")
        else:
            print("非表格内容")
            print(f"\n【内容预览】:")
            print("-" * 40)
            print(chunk_text[:300] if len(chunk_text) > 300 else chunk_text)
    else:
        print("未找到数据库记录")

conn.close()

# 验证resolve_placeholders的效果
print("\n\n" + "=" * 60)
print("验证占位符解析")
print("=" * 60)

from vector import resolve_placeholders

resolved_results = resolve_placeholders(results)
print("\n解析前后的对比:")
for i, (doc_id, text, score) in enumerate(results[:2]):
    print(f"\n[片段 {i+1}]")
    print(f"解析前: {text[:100]}...")
    resolved_text = resolved_results[i][1]
    print(f"解析后: {resolved_text[:200]}...")

测试查询: 合诚工程咨询集团的员工规模

检索结果分析

[片段 ID: 674] RRF得分: 0.0310

来源文件: 合诚工程咨询集团股份有限公司2019 年年度报告.md
非表格内容

【内容预览】:
----------------------------------------
### 六、母公司和主要子公司的员工情况

#### (一) 员工情况  

__TABLE_PLACEHOLDER_358__  

#### (二) 薪酬政策  

√适用 □不适用  

公司结合现有的行业特点和实际情况，建立了混合型的薪酬策略，并结合业务拓展持续进行薪酬激励机制优化，公司已推出了第一期限制性股票激励计划，以激励员工提高绩效，未来将继续探索多种激励措施。  

#### (三) 培训计划  

√适用 □不适用  

公司将人力资源视为第一资源，重视学习型组织的建设。为践行“工程智慧的领航者”企业愿景，实现员工技能提升、企业内部知识沉淀和后备人才队伍储备，公司成立合诚学院

[片段 ID: 1008] RRF得分: 0.0308

来源文件: 合诚工程咨询集团股份有限公司2019 年年度报告.md
表格ID: __TABLE_PLACEHOLDER_399__

【用于检索的Summary】:
----------------------------------------
该表格列出了合诚工程咨询集团股份有限公司及其各级子公司的类别、全称和简称。涵盖本公司、一级子公司和二级子公司，涉及工程检测、技术、咨询管理、设计、新材料、市政设计等多个业务领域，便于快速识别各实体简称。

【用于回答的HTML代码】:
----------------------------------------
<html><body><table><tr><td>类别</td><td>公司名称</td><td>简称</td></tr><tr><td>本公司</td><td>合诚工程咨询集团股份有限 公 司</td><td>合诚股份</td></tr><tr><td>一级子公司</td><td>厦门合诚工程检测有限公司</td><td>合诚检测</td></tr><tr><td>一级子公司</td><td>厦门合诚工程技术有限公司</td><td>合诚技术</td></tr><tr